In [1]:
# 셀 목적: 환경변수(.env)를 불러오고 LangSmith 기록용 프로젝트를 설정합니다.
# 핵심 개념: API 키 같은 비밀값은 코드에 직접 적지 않고 환경변수로 관리합니다.
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
# Langsmith 프로젝트 이름 설정
logging.langsmith("0915-Prompt")

LangSmith 추적을 시작합니다.
[프로젝트명]
0915-Prompt


In [2]:
# 셀 목적: 질문을 보낼 OpenAI 대화 모델을 준비합니다.
# 핵심 개념: ChatOpenAI() 생성만으로는 질문을 보내지 않으며 invoke() 때 API가 호출됩니다.
# llm 객체 선언
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [3]:
# 셀 목적: 나라 이름을 바꿔 넣을 수 있는 질문 양식을 만듭니다.
# 핵심 개념: {country}는 나중에 실제 값으로 바뀌는 빈칸입니다.
from langchain_core.prompts import PromptTemplate

# 템플릿 정의 {}, 는 변수로 값들어갈 자리
template = "{country}의 수도는 어디인가요?"

# 그냥 문자열로 만든 프롬프트를 Langchain이 쓸 수 있게 바꿈
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [4]:
# 셀 목적: {country} 빈칸에 대한민국을 넣어 완성된 질문을 확인합니다.
# 핵심 개념: format()은 문장만 만들고 AI는 호출하지 않습니다.
# 프롬프트 생성, 변수에 값 넣기
prompt = prompt.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [5]:
# 셀 목적: 프롬프트와 AI 모델을 체인으로 연결해 실제 답변을 받습니다.
# 핵심 개념: | 기호는 질문 양식의 결과를 다음 단계인 llm으로 전달합니다.
# template 정의
template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)

# 체인 생성
chain = prompt | llm

# country 변수에 입력된 값이 자동으로 치환되어 수행
chain.invoke("대한민국").content

'대한민국의 수도는 서울입니다.'

In [ ]:
# 셀 목적: PromptTemplate을 직접 생성하는 기본 문법을 익힙니다.
# 핵심 개념: input_variables에는 사용자가 나중에 넣어야 할 빈칸 이름을 적습니다.
template = "{country}의 수도는 어디인가요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=['country'],
)

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [7]:
# 셀 목적: 직접 만든 프롬프트에 대한민국을 넣어 결과 문장을 확인합니다.
# 이 셀은 문장만 완성하므로 API 비용이 발생하지 않습니다.
prompt.format(country="대한민국")

'대한민국의 수도는 어디인가요?'

In [ ]:
# 셀 목적: 두 나라 중 하나는 입력받고, 다른 하나는 미리 지정하는 양식을 만듭니다.
# 핵심 개념: partial_variables는 자주 쓰는 값을 기본값처럼 미리 채워 둡니다.
template = "{country1}과 {country2}의 수도는 각각 어디인가요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    # partial로 미리 country2의 값을 지정, prompt를 만들면서 지정 (Variables)
    partial_variables={
        "country2": "미국" 
    },
)

prompt

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '미국'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [9]:
# 셀 목적: country1만 입력해도 미리 지정한 미국이 함께 들어가는지 확인합니다.
# 결과 문장에는 대한민국과 미국이 자동으로 포함됩니다.
prompt.format(country1="대한민국")

'대한민국과 미국의 수도는 각각 어디인가요?'

In [11]:
# 셀 목적: 이미 만든 프롬프트의 country2 값을 캐나다로 다시 지정합니다.
# 핵심 개념: partial()은 기존 양식을 바탕으로 일부 값을 미리 채운 새 프롬프트를 만듭니다.
# 이미 만든 프롬프트에 값을 추가로 지정
prompt_partial = prompt.partial(country2="캐나다")

prompt_partial

PromptTemplate(input_variables=['country1'], input_types={}, partial_variables={'country2': '캐나다'}, template='{country1}과 {country2}의 수도는 각각 어디인가요?')

In [12]:
# 셀 목적: 새 partial 프롬프트가 대한민국과 캐나다 문장을 만드는지 확인합니다.
# format() 결과만 확인하므로 아직 AI는 호출하지 않습니다.
prompt_partial.format(country1="대한민국")

'대한민국과 캐나다의 수도는 각각 어디인가요?'

In [13]:
# 셀 목적: partial 프롬프트와 AI 모델을 연결해 실제 답변을 받습니다.
# invoke()가 실행되면 OpenAI API에 요청이 전송됩니다.
chain = prompt_partial | llm

chain.invoke("대한민국").content

'대한민국의 수도는 서울이고, 캐나다의 수도는 오타와입니다.'

In [14]:
# 셀 목적: 실행 시 country2를 호주로 직접 전달해 기존 partial 값과 다른 결과를 확인합니다.
# 직접 전달한 호주 값이 사용되어, 모델은 호주의 수도인 캔버라를 답합니다.
chain.invoke({"country1": "대한민국", "country2": "호주"}).content

'대한민국의 수도는 서울이며 호주의 수도는 캔버라입니다.'